In [1]:
!pip install tensorflow keras

In [2]:
import os
print(os.listdir('/kaggle/input'))


['datasets']


In [3]:
import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

# DISABLE XLA COMPILATION explicitly to prevent Kaggle GPU freezes
os.environ["TF_XLA_FLAGS"] = "--tf_xla_enable_xla_devices=false"
tf.config.optimizer.set_jit(False)

# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────
if os.path.exists("/kaggle/input"):
    # Kaggle Kernel path
    BASE_DIR = "/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset"
    OUTPUT_MODEL_PATH = "/kaggle/working/efficientnetb0.keras"
    TRAIN_DIR = os.path.join(BASE_DIR, "Training")
    TEST_DIR  = os.path.join(BASE_DIR, "Testing")
elif os.path.exists("/content"):
    # Google Colab path
    BASE_DIR = "/content"
    OUTPUT_MODEL_PATH = "/content/models/efficientnetb0.keras"
    TRAIN_DIR = os.path.join(BASE_DIR, "MRI images", "Training")
    TEST_DIR  = os.path.join(BASE_DIR, "MRI images", "Testing")
else:
    # Local Windows path
    BASE_DIR = r"c:\Users\Siddharth Gupta\Desktop\main_encephlo\Encephlo"
    OUTPUT_MODEL_PATH = os.path.join(BASE_DIR, "backend", "models", "efficientnetb0.keras")
    TRAIN_DIR = os.path.join(BASE_DIR, "MRI images", "Training")
    TEST_DIR  = os.path.join(BASE_DIR, "MRI images", "Testing")

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 4
SEED = 42

print("=" * 60)
print("LOADING DATA FOR EFFICIENTNET-B0")
print("=" * 60)

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    label_mode='categorical',
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    label_mode='categorical',
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    seed=SEED
)

def preprocess(image, label):
    # EfficientNet expects unscaled inputs (it has rescaling inside), but applying preprocess_input normalizes it formally.
    image = tf.keras.applications.efficientnet.preprocess_input(image)
    return image, label

train_ds = train_ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
val_ds = val_ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)

# Model Data Augmentation
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomTranslation(0.1, 0.1)
], name='data_augmentation')

print("\n" + "=" * 60)
print("BUILDING EFFICIENTNET-B0 MODEL")
print("=" * 60)

# Build the base EfficientNetB0 model
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# Phase 1: Freeze base model
base_model.trainable = False

# Build complete model enforcing training=False exactly like DenseNet
inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="input_image")
x = data_augmentation(inputs)
x = base_model(x, training=False) # CRUCIAL: Ensures BN runs in inference mode!

x = layers.GlobalAveragePooling2D(name="global_average_pooling")(x) # Creates the 1280-D vector
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax', name="classifier")(x)

model = keras.Model(inputs, outputs, name="efficientnetb0_classifier")

# Compile for Phase 1
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

print("\n" + "=" * 60)
print("PHASE 1: WARMUP TOP LAYERS")
print("=" * 60)

os.makedirs(os.path.dirname(OUTPUT_MODEL_PATH), exist_ok=True)

model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

print("\n" + "=" * 60)
print("PHASE 2: FINE-TUNING ENTIRE MODEL")
print("=" * 60)

# Unfreeze the ENTIRE model for maximum accuracy potential
base_model.trainable = True

# Ensure BatchNormalization remains frozen across the board to prevent gradient exploding
for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

# Re-compile with an EXTREMELY slow micro-tuning learning rate to crawl out of the 94% local minimum
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=20, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_acc', factor=0.2, patience=3, min_lr=1e-8, verbose=1),
    ModelCheckpoint(filepath=OUTPUT_MODEL_PATH, monitor='val_accuracy', save_best_only=True, verbose=1)
]

print("Training for high accuracy (>96%)...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=callbacks
)

print("\n" + "=" * 60)
print("EXTRACTING STRICT HEADLESS MODEL")
print("=" * 60)

# The user explicitly requires a pure headless model to avoid dynamic extraction later.
# We slice the model from input directly to the 1280-D GlobalAveragePooling layer.
gap_output = model.get_layer("global_average_pooling").output
headless_model = keras.Model(inputs=model.input, outputs=gap_output, name="efficientnet_headless_1280")

headless_model.save(OUTPUT_MODEL_PATH)

print(f"\nTraining Complete. Headless EfficientNetB0 (1280-D) saved to {OUTPUT_MODEL_PATH}")


2026-03-04 21:04:28.330393: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772658268.505670      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772658268.553997      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772658268.975536      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772658268.975572      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772658268.975575      24 computation_placer.cc:177] computation placer alr

LOADING DATA FOR EFFICIENTNET-B0
Found 5600 files belonging to 4 classes.


I0000 00:00:1772658295.665761      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Found 1600 files belonging to 4 classes.

BUILDING EFFICIENTNET-B0 MODEL
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "efficientnetb0_classifier"

...

175/175 ━━━━━━━━━━━━━━━━━━━━ 28s 88ms/step - accuracy: 0.5907 - loss: 0.9568 - val_accuracy: 0.6881 - val_loss: 0.7674
Epoch 2/10
175/175 ━━━━━━━━━━━━━━━━━━━━ 11s 65ms/step - accuracy: 0.7972 - loss: 0.5330 - val_accuracy: 0.7138 - val_loss: 0.7384
Epoch 3/10
175/175 ━━━━━━━━━━━━━━━━━━━━ 11s 65ms/step - accuracy: 0.8270 - loss: 0.4709 - val_accuracy: 0.7269 - val_loss: 0.7087
Epoch 4/10
175/175 ━━━━━━━━━━━━━━━━━━━━ 11s 65ms/step - accuracy: 0.8391 - loss: 0.4422 - val_accuracy: 0.7563 - val_loss: 0.6783
Epoch 5/10
175/175 ━━━━━━━━━━━━━━━━━━━━ 11s 64ms/step - accuracy: 0.8387 - loss: 0.4324 - val_accuracy: 0.7594 - val_loss: 0.6554
Epoch 6/10
175/175 ━━━━━━━━━━━━━━━━━━━━ 11s 65ms/step - accuracy: 0.8487 - loss: 0.4129 - val_accuracy: 0.7381 - val_loss: 0.6840
Epoch 7/10
175/175 ━━━━━━━━━━━━━━━━━━━━ 11s 65ms/step - accuracy: 0.8484 - loss: 0.3998 - val_accuracy: 0.7806 - val_loss: 0.6328
Epoch 8/10
175/175 ━━━━━━━━━━━━━━━━━━━━ 11s 65ms/step - accuracy: 0.8557 - loss: 0.3916 - val_accurac

E0000 00:00:1772658449.502623      24 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/efficientnetb0_classifier_1/efficientnetb0_1/block2b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


175/175 ━━━━━━━━━━━━━━━━━━━━ 0s 199ms/step - accuracy: 0.8707 - loss: 0.3495
Epoch 1: val_accuracy improved from -inf to 0.83438, saving model to /kaggle/working/efficientnetb0.keras


/usr/local/lib/python3.12/dist-packages/keras/src/callbacks/callback_list.py:145: UserWarning: Learning rate reduction is conditioned on metric `val_acc` which is not available. Available metrics are: accuracy,loss,val_accuracy,val_loss,learning_rate.
  callback.on_epoch_end(epoch, logs)


175/175 ━━━━━━━━━━━━━━━━━━━━ 62s 233ms/step - accuracy: 0.8707 - loss: 0.3494 - val_accuracy: 0.8344 - val_loss: 0.4695 - learning_rate: 1.0000e-05
Epoch 2/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 0s 199ms/step - accuracy: 0.8871 - loss: 0.2903
Epoch 2: val_accuracy did not improve from 0.83438
175/175 ━━━━━━━━━━━━━━━━━━━━ 37s 213ms/step - accuracy: 0.8871 - loss: 0.2903 - val_accuracy: 0.8281 - val_loss: 0.4906 - learning_rate: 1.0000e-05
Epoch 3/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 0s 198ms/step - accuracy: 0.8969 - loss: 0.2558
Epoch 3: val_accuracy improved from 0.83438 to 0.84500, saving model to /kaggle/working/efficientnetb0.keras
175/175 ━━━━━━━━━━━━━━━━━━━━ 38s 217ms/step - accuracy: 0.8970 - loss: 0.2557 - val_accuracy: 0.8450 - val_loss: 0.4538 - learning_rate: 1.0000e-05
Epoch 4/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 0s 200ms/step - accuracy: 0.9120 - loss: 0.2334
Epoch 4: val_accuracy did not improve from 0.84500
175/175 ━━━━━━━━━━━━━━━━━━━━ 37s 214ms/step - accuracy: 0.9120 - loss: 0.2334 - v